# Phase 1 : ancre BART-large (MEDIQA 2021)

Runtime : **GPU L4** (ou A100). Durée : environ 45 à 60 minutes. Toute la logique est dans le dépôt.

- **Google Drive** garde l'état du protocole (taux choisi, **registre du test**) : relancer le notebook ne peut ni perdre la sélection, ni rouvrir l'accès au test.
- Les modèles restent sur le disque éphémère de Colab. Chaque étape **reprend** là où elle s'est arrêtée : relancer une cellule ne refait pas un run terminé.
- La cellule « État » dit à tout moment ce qui est fait.

In [ ]:
REPO_URL = "https://github.com/desire-del/faithful-question-summarization.git"
REF = "main"
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['FAITHQS_STATE_DIR'] = '/content/drive/MyDrive/faithqs/state'  # ledger + protocol state
os.makedirs(os.environ['FAITHQS_STATE_DIR'], exist_ok=True)
%cd /content
# Never delete an existing clone: update it instead.
!if [ -d repo/.git ]; then git -C repo fetch -q origin && git -C repo checkout -q $REF && git -C repo pull -q --ff-only origin $REF; else git clone -q --branch $REF $REPO_URL repo; fi
%cd /content/repo
!git log --oneline -1
# Exact versions from uv.lock, except the torch/CUDA stack (Colab's build matches its driver).
!pip install -q uv
!uv export --frozen --extra train --extra eval --no-hashes --no-emit-project -o /content/req_lock.txt
!grep -v -E '^(torch|triton|nvidia-|cuda-|torchvision|torchaudio)' /content/req_lock.txt > /content/req.txt
!pip uninstall -y -q torchao
!pip install -q -r /content/req.txt
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Rebuild the splits and verify them against the versioned manifest
!faithqs data fetch

In [ ]:
# État : ce qui est fait (relance cette cellule quand tu veux)
import json, os, pathlib
state = pathlib.Path(os.environ['FAITHQS_STATE_DIR'])
sel = state / 'reports/sft/bart_lr_selection.json'
print('lr selection :', json.loads(sel.read_text())['chosen_lr'] if sel.exists() else 'not done yet')
for run in sorted(pathlib.Path('/content/faithqs-runs').glob('bart_*')):
    scores = sorted(p.name for p in (run / 'scores').glob('*.summary.json')) if (run / 'scores').exists() else []
    print(f"{run.name:22s} {'model' if (run / 'model').exists() else 'NO MODEL':9s} {scores}")
ledger = state / 'test_ledger.jsonl'
print('test ledger  :', ledger.read_text().strip() if ledger.exists() else 'empty (test never accessed)')

In [ ]:
# 1) Learning-rate search (seed 0), choice on dev ROUGE-L. Must end with 'chosen lr: ...'
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage search

In [ ]:
# 2) Seeds 1 and 2 at the chosen rate + dev predictions (greedy, beam4)
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage seeds

In [ ]:
# 3) Dev table
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage table

## ⚠️ Accès au test, une seule fois
La cellule suivante consomme l'accès `bart_reproduction` (PROTOCOL §2). Elle vérifie d'abord que les 3 modèles existent (sinon elle s'arrête **sans** toucher au test), puis journalise l'accès sur Drive et le ferme. Une seconde tentative est refusée.

In [ ]:
# 4) BART reproduction on the official MEDIQA 2021 test set (logged access)
!python scripts/run_sft_protocol.py --env colab --model bart_large --sft-config configs/sft/bart_large.yaml --tag bart --stage test

In [ ]:
# 5) Artifacts: run manifests, predictions, scores, protocol reports and the test ledger (no weights)
import pathlib, zipfile, os
runs = pathlib.Path('/content/faithqs-runs')
state = pathlib.Path(os.environ['FAITHQS_STATE_DIR'])
archive = pathlib.Path('/content/bart_anchor_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in runs.rglob('*'):
        if path.is_file() and not {'model', 'checkpoints'} & set(path.parts):
            z.write(path, 'runs/' + str(path.relative_to(runs)))
    for path in (state / 'reports').rglob('*'):
        if path.is_file():
            z.write(path, 'reports/' + str(path.relative_to(state / 'reports')))
    if (state / 'test_ledger.jsonl').exists():
        z.write(state / 'test_ledger.jsonl', 'reports/test_ledger.jsonl')
print(archive, round(archive.stat().st_size / 1e6, 2), 'MB')
!cp /content/bart_anchor_artifacts.zip /content/drive/MyDrive/faithqs/
from google.colab import files
files.download(str(archive))

In [ ]:
# 6) Summary to paste back
import json, os, pathlib
rep_dir = pathlib.Path(os.environ['FAITHQS_STATE_DIR']) / 'reports/sft'
print((rep_dir / 'bart_dev.md').read_text())
rep = json.loads((rep_dir / 'bart_test_reproduction.json').read_text())
print('chosen lr', rep['lr'])
print(json.dumps(rep['vs_focusmed_bart'], indent=1))